In [1]:
!pip install spacy scikit-learn

In [2]:
import spacy
import pandas as pd

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
data = [
    {
        "text": "Apple is planning to open a new office in Bengaluru next year.",
        "entities": [("Apple", "ORG"), ("Bengaluru", "GPE")]
    },
    {
        "text": "Elon Musk is the CEO of Tesla and SpaceX.",
        "entities": [("Elon Musk", "PERSON"), ("Tesla", "ORG"), ("SpaceX", "ORG")]
    },
    {
        "text": "Google announced a new AI research lab in London.",
        "entities": [("Google", "ORG"), ("London", "GPE")]
    },
    {
        "text": "Microsoft acquired LinkedIn for 26 billion dollars.",
        "entities": [("Microsoft", "ORG"), ("LinkedIn", "ORG")]
    },
    {
        "text": "Prime Minister Narendra Modi visited the United States.",
        "entities": [("Narendra Modi", "PERSON"), ("United States", "GPE")]
    },
    {
        "text": "Amazon has its headquarters in Seattle.",
        "entities": [("Amazon", "ORG"), ("Seattle", "GPE")]
    },
    {
        "text": "Meta launched a new feature on Instagram last week.",
        "entities": [("Meta", "ORG"), ("Instagram", "ORG")]
    },
    {
        "text": "The World Health Organization is based in Geneva.",
        "entities": [("World Health Organization", "ORG"), ("Geneva", "GPE")]
    },
    {
        "text": "Sundar Pichai spoke at the Google I/O event.",
        "entities": [("Sundar Pichai", "PERSON"), ("Google", "ORG")]
    },
    {
        "text": "India will host the G20 summit in New Delhi.",
        "entities": [("India", "GPE"), ("G20", "ORG"), ("New Delhi", "GPE")]
    }
]

df = pd.DataFrame(data)
df

,text,entities
0,Apple is planning to open a new office in Beng...,"[(Apple, ORG), (Bengaluru, GPE)]"
1,Elon Musk is the CEO of Tesla and SpaceX.,"[(Elon Musk, PERSON), (Tesla, ORG), (SpaceX, O..."
2,Google announced a new AI research lab in London.,"[(Google, ORG), (London, GPE)]"
3,Microsoft acquired LinkedIn for 26 billion dol...,"[(Microsoft, ORG), (LinkedIn, ORG)]"
4,Prime Minister Narendra Modi visited the Unite...,"[(Narendra Modi, PERSON), (United States, GPE)]"
5,Amazon has its headquarters in Seattle.,"[(Amazon, ORG), (Seattle, GPE)]"
6,Meta launched a new feature on Instagram last ...,"[(Meta, ORG), (Instagram, ORG)]"
7,The World Health Organization is based in Geneva.,"[(World Health Organization, ORG), (Geneva, GPE)]"
8,Sundar Pichai spoke at the Google I/O event.,"[(Sundar Pichai, PERSON), (Google, ORG)]"
9,India will host the G20 summit in New Delhi.,"[(India, GPE), (G20, ORG), (New Delhi, GPE)]"


In [5]:
def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

df["predicted_entities"] = df["text"].apply(extract_entities)
df

,text,entities,predicted_entities
0,Apple is planning to open a new office in Beng...,"[(Apple, ORG), (Bengaluru, GPE)]","[(Apple, ORG), (Bengaluru, GPE), (next year, D..."
1,Elon Musk is the CEO of Tesla and SpaceX.,"[(Elon Musk, PERSON), (Tesla, ORG), (SpaceX, O...","[(Elon Musk, PERSON), (Tesla, ORG)]"
2,Google announced a new AI research lab in London.,"[(Google, ORG), (London, GPE)]","[(Google, ORG), (AI, GPE), (London, GPE)]"
3,Microsoft acquired LinkedIn for 26 billion dol...,"[(Microsoft, ORG), (LinkedIn, ORG)]","[(Microsoft, ORG), (LinkedIn, ORG), (26 billio..."
4,Prime Minister Narendra Modi visited the Unite...,"[(Narendra Modi, PERSON), (United States, GPE)]","[(Narendra Modi, PERSON), (the United States, ..."
5,Amazon has its headquarters in Seattle.,"[(Amazon, ORG), (Seattle, GPE)]","[(Amazon, ORG), (Seattle, GPE)]"
6,Meta launched a new feature on Instagram last ...,"[(Meta, ORG), (Instagram, ORG)]","[(Meta, ORG), (Instagram, ORG), (last week, DA..."
7,The World Health Organization is based in Geneva.,"[(World Health Organization, ORG), (Geneva, GPE)]","[(The World Health Organization, ORG), (Geneva..."
8,Sundar Pichai spoke at the Google I/O event.,"[(Sundar Pichai, PERSON), (Google, ORG)]","[(Sundar Pichai, PERSON), (Google, ORG)]"
9,India will host the G20 summit in New Delhi.,"[(India, GPE), (G20, ORG), (New Delhi, GPE)]","[(India, GPE), (G20, ORG), (New Delhi, GPE)]"


In [6]:
true_labels = []
pred_labels = []

for i in range(len(df)):
    true = df.loc[i, "entities"]
    pred = df.loc[i, "predicted_entities"]

    true_dict = {text: label for text, label in true}
    pred_dict = {text: label for text, label in pred}

    for ent_text in true_dict:
        true_labels.append(true_dict[ent_text])
        pred_labels.append(pred_dict.get(ent_text, "O"))  # O = Not detected

In [7]:
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, average='weighted', zero_division=0)
recall = recall_score(true_labels, pred_labels, average='weighted', zero_division=0)
f1 = f1_score(true_labels, pred_labels, average='weighted', zero_division=0)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.8636363636363636
Precision: 1.0
Recall   : 0.8636363636363636
F1 Score : 0.9259376986649713


In [8]:
print(classification_report(true_labels, pred_labels, zero_division=0))

              precision    recall  f1-score   support

         GPE       1.00      0.86      0.92         7
           O       0.00      0.00      0.00         0
         ORG       1.00      0.83      0.91        12
      PERSON       1.00      1.00      1.00         3

    accuracy                           0.86        22
   macro avg       0.75      0.67      0.71        22
weighted avg       1.00      0.86      0.93        22



In [10]:
df.to_csv("ner_results.csv", index=False)
print("NER results saved successfully!")

NER results saved successfully!
